<a href="https://colab.research.google.com/github/sachin23-an/Quant-Finance-Projects/blob/main/Dynamic_Portfolio_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dynamic Portfolio Optimization: Maximizing Returns and Minimizing Risk

### Project Overview

This project details a comprehensive approach to dynamic portfolio optimization, focusing on maximizing returns while minimizing risk within the Indian equity market. We leverage historical stock data to construct, analyze, and backtest various portfolio strategies, including Monte Carlo simulations and formal mathematical optimization.

The core objective is to identify optimal portfolio allocations that deliver superior risk-adjusted returns, providing a robust framework for data-driven investment decisions. The analysis includes detailed calculations of returns, volatility, inter-asset dynamics (covariance and correlation), and a thorough assessment of historical performance and potential drawdowns against market benchmarks.

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

### Asset Data Acquisition
This section details the process of collecting historical price data for a selected basket of securities from the Indian market. The data spans a three-year period, providing a robust foundation for our portfolio analysis.

In [ ]:
tickers = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'INFY.NS',
    'ITC.NS', 'LT.NS', 'AXISBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'KOTAKBANK.NS'
]

end_date = pd.Timestamp.now()
start_date = end_date - pd.DateOffset(years=3)

print(f"Downloading data for {len(tickers)} securities from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

# Removed ['Adj Close'] as yf.download with auto_adjust=True (default) returns adjusted close directly.
data = yf.download(tickers, start=start_date, end=end_date)

# Select only the 'Close' prices from the MultiIndex DataFrame
data = data['Close']

# Clean data by filling missing values
data = data.ffill().bfill() # Forward fill then backfill any remaining NaNs

if data.isnull().sum().sum() > 0:
    print("Warning: There are still missing values after cleaning.")

print("\nFirst 5 rows of the adjusted close prices:")
display(data.head())

[                       0%                       ]

[*********************100%***********************]  11 of 11 completed


First 5 rows of the adjusted close prices:


Ticker,AXISBANK.NS,BHARTIARTL.NS,HDFCBANK.NS,ICICIBANK.NS,INFY.NS,ITC.NS,KOTAKBANK.NS,LT.NS,RELIANCE.NS,SBIN.NS,TCS.NS
Date,,,,,,,,,,,
2023-04-17,862.265686,745.751465,801.938232,879.525085,1154.021362,345.076660,377.867889,2151.948486,1080.732910,514.498169,2887.303711
2023-04-18,861.068848,745.849670,798.545959,875.182617,1154.984375,343.653687,376.502075,2154.475830,1068.361938,516.247742,2879.256836
2023-04-19,871.092407,750.660034,801.481140,870.352234,1130.084351,343.869293,375.824188,2157.391602,1073.680054,510.998779,2841.412598
2023-04-20,866.454651,758.170044,804.464355,872.791870,1122.013794,345.205994,375.644714,2170.172852,1070.963867,515.254761,2855.391602
2023-04-21,861.916626,751.199951,805.763550,864.253235,1125.636230,352.061768,377.568817,2153.114990,1072.310547,513.694275,2906.939209


###  Return Calculation
Daily returns, annualized returns, and annualized volatility were calculated, assuming 252 trading days in a year.

In [ ]:
trading_days_per_year = 252

# Calculate daily returns
daily_returns = data.pct_change().dropna()

# Calculate annualized returns
annualized_returns = daily_returns.mean() * trading_days_per_year

# Calculate annualized volatility
annualized_volatility = daily_returns.std() * np.sqrt(trading_days_per_year)

print("\nAnnualized Returns:")
display(annualized_returns)

print("\nAnnualized Volatility:")
display(annualized_volatility)


Annualized Returns:


,0
Ticker,
AXISBANK.NS,0.180722
BHARTIARTL.NS,0.331583
HDFCBANK.NS,0.022445
ICICIBANK.NS,0.162278
INFY.NS,0.071014
ITC.NS,-0.028062
KOTAKBANK.NS,0.028512
LT.NS,0.250340
RELIANCE.NS,0.095482



Annualized Volatility:


,0
Ticker,
AXISBANK.NS,0.231137
BHARTIARTL.NS,0.204394
HDFCBANK.NS,0.196199
ICICIBANK.NS,0.184176
INFY.NS,0.240811
ITC.NS,0.185371
KOTAKBANK.NS,0.219529
LT.NS,0.254242
RELIANCE.NS,0.205896


### Inter-Asset Dynamics: Covariance and Correlation
Effective portfolio diversification hinges on understanding how different assets interact. This section quantifies these relationships through annualized covariance and correlation matrices, revealing the interconnectedness and potential hedging benefits among the chosen securities.

In [ ]:
covariance_matrix = daily_returns.cov() * trading_days_per_year
correlation_matrix = daily_returns.corr()

print("\nCovariance Matrix (Annualized):")
display(covariance_matrix)

print("\nCorrelation Matrix:")
display(correlation_matrix)


Covariance Matrix (Annualized):


Ticker,AXISBANK.NS,BHARTIARTL.NS,HDFCBANK.NS,ICICIBANK.NS,INFY.NS,ITC.NS,KOTAKBANK.NS,LT.NS,RELIANCE.NS,SBIN.NS,TCS.NS
Ticker,,,,,,,,,,,
AXISBANK.NS,0.053424,0.011377,0.021039,0.023290,0.010090,0.009776,0.018579,0.024243,0.017631,0.027065,0.008113
BHARTIARTL.NS,0.011377,0.041777,0.009924,0.012607,0.011637,0.008175,0.014091,0.014221,0.013986,0.011821,0.008720
HDFCBANK.NS,0.021039,0.009924,0.038494,0.017557,0.008781,0.007219,0.018634,0.020388,0.014912,0.017818,0.006411
ICICIBANK.NS,0.023290,0.012607,0.017557,0.033921,0.008636,0.008179,0.017214,0.018950,0.012127,0.019361,0.006323
INFY.NS,0.010090,0.011637,0.008781,0.008636,0.057990,0.005764,0.008133,0.014344,0.011489,0.008562,0.036497
ITC.NS,0.009776,0.008175,0.007219,0.008179,0.005764,0.034362,0.011275,0.010626,0.011457,0.009885,0.005357
KOTAKBANK.NS,0.018579,0.014091,0.018634,0.017214,0.008133,0.011275,0.048193,0.019361,0.015023,0.017903,0.006466
LT.NS,0.024243,0.014221,0.020388,0.018950,0.014344,0.010626,0.019361,0.064639,0.023852,0.029980,0.013887
RELIANCE.NS,0.017631,0.013986,0.014912,0.012127,0.011489,0.011457,0.015023,0.023852,0.042393,0.022123,0.010873



Correlation Matrix:


Ticker,AXISBANK.NS,BHARTIARTL.NS,HDFCBANK.NS,ICICIBANK.NS,INFY.NS,ITC.NS,KOTAKBANK.NS,LT.NS,RELIANCE.NS,SBIN.NS,TCS.NS
Ticker,,,,,,,,,,,
AXISBANK.NS,1.000000,0.240825,0.463943,0.547089,0.181272,0.228165,0.366161,0.412544,0.370465,0.492730,0.168245
BHARTIARTL.NS,0.240825,1.000000,0.247460,0.334890,0.236431,0.215761,0.314042,0.273672,0.332338,0.243360,0.204494
HDFCBANK.NS,0.463943,0.247460,1.000000,0.485871,0.185856,0.198481,0.432639,0.408728,0.369132,0.382156,0.156623
ICICIBANK.NS,0.547089,0.334890,0.485871,1.000000,0.194727,0.239568,0.425745,0.404699,0.319784,0.442365,0.164564
INFY.NS,0.181272,0.236431,0.185856,0.194727,1.000000,0.129115,0.153844,0.234282,0.231711,0.149617,0.726494
ITC.NS,0.228165,0.215761,0.198481,0.239568,0.129115,1.000000,0.277055,0.225471,0.300178,0.224390,0.138533
KOTAKBANK.NS,0.366161,0.314042,0.432639,0.425745,0.153844,0.277055,1.000000,0.346886,0.332368,0.343170,0.141194
LT.NS,0.412544,0.273672,0.408728,0.404699,0.234282,0.225471,0.346886,1.000000,0.455654,0.496200,0.261827
RELIANCE.NS,0.370465,0.332338,0.369132,0.319784,0.231711,0.300178,0.332368,0.455654,1.000000,0.452138,0.253134


### Simulating Investment Horizons: Monte Carlo Portfolio Generation
To explore a wide spectrum of investment possibilities, we utilize a Monte Carlo simulation. By generating 5,000 random portfolio allocations, we calculate their prospective annual return, volatility, and Sharpe Ratio (assuming a 0% risk-free rate). This simulation provides a rich dataset for identifying promising portfolio configurations.

In [ ]:
num_portfolios = 5000
risk_free_rate = 0.0 # Assuming 0% risk-free rate as per prompt

portfolio_returns = []
portfolio_volatilities = []
portfolio_sharpe_ratios = []
portfolio_weights = []

num_assets = len(tickers)

for _ in range(num_portfolios):
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    portfolio_weights.append(weights)

    # Expected annual return
    p_return = np.sum(weights * annualized_returns)
    portfolio_returns.append(p_return)

    # Portfolio volatility
    p_volatility = np.sqrt(np.dot(weights.T, np.dot(covariance_matrix, weights)))
    portfolio_volatilities.append(p_volatility)

    # Sharpe ratio
    sharpe_ratio = (p_return - risk_free_rate) / p_volatility
    portfolio_sharpe_ratios.append(sharpe_ratio)

portfolio_results_df = pd.DataFrame({
    'Return': portfolio_returns,
    'Volatility': portfolio_volatilities,
    'Sharpe Ratio': portfolio_sharpe_ratios
})

# Add individual asset weights
for i, ticker in enumerate(tickers):
    portfolio_results_df[ticker + ' Weight'] = [w[i] for w in portfolio_weights]

print("\nFirst 5 rows of Monte Carlo simulation results:")
display(portfolio_results_df.head())


First 5 rows of Monte Carlo simulation results:


,Return,Volatility,Sharpe Ratio,RELIANCE.NS Weight,TCS.NS Weight,HDFCBANK.NS Weight,ICICIBANK.NS Weight,INFY.NS Weight,ITC.NS Weight,LT.NS Weight,AXISBANK.NS Weight,SBIN.NS Weight,BHARTIARTL.NS Weight,KOTAKBANK.NS Weight
0,0.119753,0.141245,0.847837,0.057937,0.034825,0.022830,0.031792,0.152056,0.026437,0.105152,0.132967,0.157904,0.121124,0.156976
1,0.095179,0.131156,0.725690,0.065996,0.036388,0.130479,0.125729,0.129315,0.117139,0.106230,0.120971,0.003683,0.038650,0.125423
2,0.127180,0.136643,0.930747,0.121699,0.045253,0.015909,0.133137,0.028242,0.128875,0.141778,0.141295,0.069672,0.091182,0.082957
3,0.094119,0.131133,0.717741,0.080190,0.003740,0.060423,0.118609,0.088291,0.102810,0.119629,0.013108,0.126878,0.138253,0.148070
4,0.105435,0.126610,0.832752,0.107965,0.159064,0.151361,0.039717,0.146163,0.143588,0.052842,0.016526,0.118760,0.004243,0.059772


In [ ]:
max_sharpe_portfolio = portfolio_results_df.loc[portfolio_results_df['Sharpe Ratio'].idxmax()]
min_volatility_portfolio = portfolio_results_df.loc[portfolio_results_df['Volatility'].idxmin()]

# Extract weights for easy access
max_sharpe_weights = max_sharpe_portfolio[[col for col in max_sharpe_portfolio.index if 'Weight' in col]].values
min_volatility_weights = min_volatility_portfolio[[col for col in min_volatility_portfolio.index if 'Weight' in col]].values

max_sharpe_weights_df = pd.Series(max_sharpe_weights, index=tickers, name='Max Sharpe Weights')
min_volatility_weights_df = pd.Series(min_volatility_weights, index=tickers, name='Min Volatility Weights')

print("Monte Carlo Maximum Sharpe Ratio Portfolio (Summary):")
display(max_sharpe_portfolio)
print("\nMonte Carlo Minimum Volatility Portfolio (Summary):")
display(min_volatility_portfolio)

Monte Carlo Maximum Sharpe Ratio Portfolio (Summary):


,1126
Return,0.196003
Volatility,0.140304
Sharpe Ratio,1.396992
RELIANCE.NS Weight,0.144284
TCS.NS Weight,0.269457
HDFCBANK.NS Weight,0.044308
ICICIBANK.NS Weight,0.060420
INFY.NS Weight,0.130128
ITC.NS Weight,0.011223
LT.NS Weight,0.006781



Monte Carlo Minimum Volatility Portfolio (Summary):


,2993
Return,0.096538
Volatility,0.122711
Sharpe Ratio,0.786714
RELIANCE.NS Weight,0.033702
TCS.NS Weight,0.158464
HDFCBANK.NS Weight,0.114265
ICICIBANK.NS Weight,0.133317
INFY.NS Weight,0.064975
ITC.NS Weight,0.169898
LT.NS Weight,0.091856


### Refining Optimality: Formal Portfolio Optimization
While Monte Carlo simulations offer valuable approximations, formal mathematical optimization provides precise solutions for ideal portfolios. In this section, we apply rigorous optimization techniques to identify the exact portfolio compositions that either maximize the Sharpe Ratio or minimize portfolio volatility, thus confirming and refining our earlier simulated findings.

In [ ]:
# Define functions for optimization
def portfolio_return(weights):
    return np.sum(weights * annualized_returns)

def portfolio_volatility(weights):
    return np.sqrt(np.dot(weights.T, np.dot(covariance_matrix, weights)))

def neg_sharpe_ratio(weights):
    return -(portfolio_return(weights) - risk_free_rate) / portfolio_volatility(weights)

# Constraints and Bounds
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for asset in range(num_assets))

initial_weights = np.array([1./num_assets] * num_assets)

# --- Minimize Volatility ---
result_min_volatility = minimize(
    portfolio_volatility, initial_weights, method='SLSQP',
    bounds=bounds, constraints=constraints
)

optimal_volatility_weights = result_min_volatility.x
optimal_volatility_return = portfolio_return(optimal_volatility_weights)
optimal_volatility_std = portfolio_volatility(optimal_volatility_weights)

optimal_volatility_portfolio = pd.Series({
    'Return': optimal_volatility_return,
    'Volatility': optimal_volatility_std,
    'Sharpe Ratio': (optimal_volatility_return - risk_free_rate) / optimal_volatility_std
})

# --- Maximize Sharpe Ratio (Minimize Negative Sharpe Ratio) ---
result_max_sharpe = minimize(
    neg_sharpe_ratio, initial_weights, method='SLSQP',
    bounds=bounds, constraints=constraints
)

optimal_sharpe_weights = result_max_sharpe.x
optimal_sharpe_return = portfolio_return(optimal_sharpe_weights)
optimal_sharpe_std = portfolio_volatility(optimal_sharpe_weights)

optimal_sharpe_portfolio = pd.Series({
    'Return': optimal_sharpe_return,
    'Volatility': optimal_sharpe_std,
    'Sharpe Ratio': (optimal_sharpe_return - risk_free_rate) / optimal_sharpe_std
})

print("\nOptimized Maximum Sharpe Ratio Portfolio (Formal Optimization):")
display(optimal_sharpe_portfolio)

print("\nOptimized Minimum Volatility Portfolio (Formal Optimization):")
display(optimal_volatility_portfolio)

optimal_sharpe_weights_df = pd.Series(optimal_sharpe_weights, index=tickers, name='Optimal Max Sharpe Weights')
optimal_volatility_weights_df = pd.Series(optimal_volatility_weights, index=tickers, name='Optimal Min Volatility Weights')

print("\nOptimal Maximum Sharpe Ratio Portfolio Weights:")
display(optimal_sharpe_weights_df[optimal_sharpe_weights_df > 0.0001].round(4))

print("\nOptimal Minimum Volatility Portfolio Weights:")
display(optimal_volatility_weights_df[optimal_volatility_weights_df > 0.0001].round(4))


Optimized Maximum Sharpe Ratio Portfolio (Formal Optimization):


,0
Return,0.309061
Volatility,0.169585
Sharpe Ratio,1.822452



Optimized Minimum Volatility Portfolio (Formal Optimization):


,0
Return,0.074776
Volatility,0.119973
Sharpe Ratio,0.623273



Optimal Maximum Sharpe Ratio Portfolio Weights:


,Optimal Max Sharpe Weights
TCS.NS,0.6279
ICICIBANK.NS,0.0014
AXISBANK.NS,0.0931
BHARTIARTL.NS,0.2776



Optimal Minimum Volatility Portfolio Weights:


,Optimal Min Volatility Weights
TCS.NS,0.1350
HDFCBANK.NS,0.1372
ICICIBANK.NS,0.1504
INFY.NS,0.0077
ITC.NS,0.2503
LT.NS,0.0394
SBIN.NS,0.0561
BHARTIARTL.NS,0.0232
KOTAKBANK.NS,0.2006


In [ ]:
fig = px.scatter(
    portfolio_results_df,
    x='Volatility',
    y='Return',
    color='Sharpe Ratio',
    hover_name='Sharpe Ratio',
    title='Efficient Frontier with Monte Carlo Simulation and Formal Optimization',
    labels={'Volatility': 'Annualized Volatility', 'Return': 'Annualized Return'},
    color_continuous_scale=px.colors.sequential.Viridis
)

# Highlight Monte Carlo Maximum Sharpe Ratio Portfolio
fig.add_trace(go.Scatter(
    mode='markers',
    x=[max_sharpe_portfolio['Volatility']],
    y=[max_sharpe_portfolio['Return']],
    marker=dict(color='red', size=15, symbol='star'),
    name='MC Max Sharpe Portfolio'
))

# Highlight Monte Carlo Minimum Volatility Portfolio
fig.add_trace(go.Scatter(
    mode='markers',
    x=[min_volatility_portfolio['Volatility']],
    y=[min_volatility_portfolio['Return']],
    marker=dict(color='blue', size=15, symbol='star'),
    name='MC Min Volatility Portfolio'
))

# Highlight Formally Optimized Maximum Sharpe Ratio Portfolio
fig.add_trace(go.Scatter(
    mode='markers',
    x=[optimal_sharpe_portfolio['Volatility']],
    y=[optimal_sharpe_portfolio['Return']],
    marker=dict(color='green', size=18, symbol='star-open', line=dict(width=2, color='DarkSlateGrey')),
    name='Optimized Max Sharpe Portfolio'
))

# Highlight Formally Optimized Minimum Volatility Portfolio
fig.add_trace(go.Scatter(
    mode='markers',
    x=[optimal_volatility_portfolio['Volatility']],
    y=[optimal_volatility_portfolio['Return']],
    marker=dict(color='purple', size=18, symbol='star-open', line=dict(width=2, color='DarkSlateGrey')),
    name='Optimized Min Volatility Portfolio'
))

fig.update_layout(xaxis_title='Portfolio Volatility (Std. Dev.)', yaxis_title='Portfolio Expected Return')
fig.show()

In [ ]:
print("### Optimal Portfolio Weights ###")
print("\nMonte Carlo Maximum Sharpe Ratio Portfolio Weights (only showing > 0.01%):")
display(max_sharpe_weights_df[max_sharpe_weights_df > 0.0001].round(4))

print("\nFormally Optimized Maximum Sharpe Ratio Portfolio Weights (only showing > 0.01%):")
display(optimal_sharpe_weights_df[optimal_sharpe_weights_df > 0.0001].round(4))

print("\nMonte Carlo Minimum Volatility Portfolio Weights (only showing > 0.01%):")
display(min_volatility_weights_df[min_volatility_weights_df > 0.0001].round(4))

print("\nFormally Optimized Minimum Volatility Portfolio Weights (only showing > 0.01%):")
display(optimal_volatility_weights_df[optimal_volatility_weights_df > 0.0001].round(4))

# Pie chart for Formally Optimized Max Sharpe Portfolio Weights
fig_optimal_sharpe_pie = px.pie(
    names=optimal_sharpe_weights_df.index,
    values=optimal_sharpe_weights_df.values,
    title='Formally Optimized Maximum Sharpe Ratio Portfolio Allocation',
    labels={'names': 'Asset', 'values': 'Weight'}
)
fig_optimal_sharpe_pie.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#000000', width=1)))
fig_optimal_sharpe_pie.show()

# Pie chart for Formally Optimized Min Volatility Portfolio Weights
fig_optimal_min_vol_pie = px.pie(
    names=optimal_volatility_weights_df.index,
    values=optimal_volatility_weights_df.values,
    title='Formally Optimized Minimum Volatility Portfolio Allocation',
    labels={'names': 'Asset', 'values': 'Weight'}
)
fig_optimal_min_vol_pie.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#000000', width=1)))
fig_optimal_min_vol_pie.show()

print("\n### Performance Metrics ###")
print("\nMonte Carlo Maximum Sharpe Ratio Portfolio:")
print(f"  Expected Annual Return: {max_sharpe_portfolio['Return']:.2%}")
print(f"  Annualized Volatility: {max_sharpe_portfolio['Volatility']:.2%}")
print(f"  Sharpe Ratio: {max_sharpe_portfolio['Sharpe Ratio']:.4f}")

print("\nFormally Optimized Maximum Sharpe Ratio Portfolio:")
print(f"  Expected Annual Return: {optimal_sharpe_portfolio['Return']:.2%}")
print(f"  Annualized Volatility: {optimal_sharpe_portfolio['Volatility']:.2%}")
print(f"  Sharpe Ratio: {optimal_sharpe_portfolio['Sharpe Ratio']:.4f}")

print("\nMonte Carlo Minimum Volatility Portfolio:")
print(f"  Expected Annual Return: {min_volatility_portfolio['Return']:.2%}")
print(f"  Annualized Volatility: {min_volatility_portfolio['Volatility']:.2%}")
print(f"  Sharpe Ratio: {min_volatility_portfolio['Sharpe Ratio']:.4f}")

print("\nFormally Optimized Minimum Volatility Portfolio:")
print(f"  Expected Annual Return: {optimal_volatility_portfolio['Return']:.2%}")
print(f"  Annualized Volatility: {optimal_volatility_portfolio['Volatility']:.2%}")
print(f"  Sharpe Ratio: {optimal_volatility_portfolio['Sharpe Ratio']:.4f}")

### Optimal Portfolio Weights ###

Monte Carlo Maximum Sharpe Ratio Portfolio Weights (only showing > 0.01%):


,Max Sharpe Weights
RELIANCE.NS,0.1443
TCS.NS,0.2695
HDFCBANK.NS,0.0443
ICICIBANK.NS,0.0604
INFY.NS,0.1301
ITC.NS,0.0112
LT.NS,0.0068
AXISBANK.NS,0.1308
SBIN.NS,0.0386
BHARTIARTL.NS,0.0921



Formally Optimized Maximum Sharpe Ratio Portfolio Weights (only showing > 0.01%):


,Optimal Max Sharpe Weights
TCS.NS,0.6279
ICICIBANK.NS,0.0014
AXISBANK.NS,0.0931
BHARTIARTL.NS,0.2776



Monte Carlo Minimum Volatility Portfolio Weights (only showing > 0.01%):


,Min Volatility Weights
RELIANCE.NS,0.0337
TCS.NS,0.1585
HDFCBANK.NS,0.1143
ICICIBANK.NS,0.1333
INFY.NS,0.0650
ITC.NS,0.1699
LT.NS,0.0919
AXISBANK.NS,0.0422
SBIN.NS,0.0261
BHARTIARTL.NS,0.0048



Formally Optimized Minimum Volatility Portfolio Weights (only showing > 0.01%):


,Optimal Min Volatility Weights
TCS.NS,0.1350
HDFCBANK.NS,0.1372
ICICIBANK.NS,0.1504
INFY.NS,0.0077
ITC.NS,0.2503
LT.NS,0.0394
SBIN.NS,0.0561
BHARTIARTL.NS,0.0232
KOTAKBANK.NS,0.2006



### Performance Metrics ###

Monte Carlo Maximum Sharpe Ratio Portfolio:
  Expected Annual Return: 19.60%
  Annualized Volatility: 14.03%
  Sharpe Ratio: 1.3970

Formally Optimized Maximum Sharpe Ratio Portfolio:
  Expected Annual Return: 30.91%
  Annualized Volatility: 16.96%
  Sharpe Ratio: 1.8225

Monte Carlo Minimum Volatility Portfolio:
  Expected Annual Return: 9.65%
  Annualized Volatility: 12.27%
  Sharpe Ratio: 0.7867

Formally Optimized Minimum Volatility Portfolio:
  Expected Annual Return: 7.48%
  Annualized Volatility: 12.00%
  Sharpe Ratio: 0.6233


### Performance Validation: Historical Portfolio Backtest
To assess the practical efficacy of our optimized portfolios, we conduct a historical backtest. This simulation tracks the growth of an initial investment of ₹1,100,000 over the past three years for the Maximum Sharpe, Minimum Volatility, and an equally weighted portfolio, alongside a relevant market benchmark. The resulting growth trajectories provide a clear picture of their historical performance.

In [ ]:
initial_investment = 1100000

# Equal weight portfolio
equal_weights = np.array([1/num_assets] * num_assets)

# Calculate daily returns for each portfolio
max_sharpe_daily_returns = daily_returns.dot(max_sharpe_weights)
min_volatility_daily_returns = daily_returns.dot(min_volatility_weights)
equal_weight_daily_returns = daily_returns.dot(equal_weights)

# Download benchmark data (Nifty 50)
benchmark_ticker = '^NSEI'
benchmark_data = yf.download(benchmark_ticker, start=start_date, end=end_date)['Close'].squeeze()
benchmark_daily_returns = benchmark_data.pct_change().dropna()
benchmark_cumulative_returns = (1 + benchmark_daily_returns).cumprod()

# Calculate cumulative returns
max_sharpe_cumulative_returns = (1 + max_sharpe_daily_returns).cumprod()
min_volatility_cumulative_returns = (1 + min_volatility_daily_returns).cumprod()
equal_weight_cumulative_returns = (1 + equal_weight_daily_returns).cumprod()

# Calculate portfolio values over time
max_sharpe_portfolio_value = initial_investment * max_sharpe_cumulative_returns
min_volatility_portfolio_value = initial_investment * min_volatility_cumulative_returns
equal_weight_portfolio_value = initial_investment * equal_weight_cumulative_returns
benchmark_portfolio_value = initial_investment * benchmark_cumulative_returns

# Create a DataFrame for plotting
portfolio_growth_df = pd.DataFrame({
    'Max Sharpe': max_sharpe_portfolio_value,
    'Min Volatility': min_volatility_portfolio_value,
    'Equal Weight': equal_weight_portfolio_value,
    'Nifty 50 Benchmark': benchmark_portfolio_value
}).dropna()

# Plotting portfolio growth
fig = px.line(
    portfolio_growth_df,
    x=portfolio_growth_df.index,
    y=portfolio_growth_df.columns,
    title=f'Portfolio Growth Simulation (Initial Investment: ₹{initial_investment:,.2f})',
    labels={'value': 'Portfolio Value (₹)', 'index': 'Date'},
    line_group='variable',
    hover_name='variable'
)

fig.update_layout(
    yaxis_title='Portfolio Value (₹)',
    xaxis_title='Date',
    legend_title='Portfolio Type'
)

fig.show()

[*********************100%***********************]  1 of 1 completed


### Risk Assessment: Portfolio Drawdowns

Beyond returns and volatility, understanding drawdowns is crucial for risk management. This section quantifies the maximum historical decline from a peak in portfolio value for each of the optimized portfolios and the benchmark, offering insights into their downside risk.

In [ ]:
# Calculate running maximum for each portfolio's cumulative returns
max_sharpe_peak = max_sharpe_cumulative_returns.expanding(min_periods=1).max()
min_volatility_peak = min_volatility_cumulative_returns.expanding(min_periods=1).max()
equal_weight_peak = equal_weight_cumulative_returns.expanding(min_periods=1).max()
benchmark_peak = benchmark_cumulative_returns.expanding(min_periods=1).max()

# Calculate drawdowns
max_sharpe_drawdown = (max_sharpe_cumulative_returns - max_sharpe_peak) / max_sharpe_peak
min_volatility_drawdown = (min_volatility_cumulative_returns - min_volatility_peak) / min_volatility_peak
equal_weight_drawdown = (equal_weight_cumulative_returns - equal_weight_peak) / equal_weight_peak
benchmark_drawdown = (benchmark_cumulative_returns - benchmark_peak) / benchmark_peak

# Create a DataFrame for plotting drawdowns
portfolio_drawdown_df = pd.DataFrame({
    'Max Sharpe': max_sharpe_drawdown,
    'Min Volatility': min_volatility_drawdown,
    'Equal Weight': equal_weight_drawdown,
    'Nifty 50 Benchmark': benchmark_drawdown
}).dropna()

# Plotting drawdowns
fig_drawdown = px.line(
    portfolio_drawdown_df,
    x=portfolio_drawdown_df.index,
    y=portfolio_drawdown_df.columns,
    title='Portfolio Drawdowns',
    labels={'value': 'Drawdown', 'index': 'Date'},
    line_group='variable',
    hover_name='variable'
)

fig_drawdown.update_layout(
    yaxis_title='Drawdown (%)',
    xaxis_title='Date',
    legend_title='Portfolio Type',
    yaxis_tickformat='.2%'
)

fig_drawdown.show()


### Conclusion

This project successfully demonstrated a comprehensive workflow for dynamic portfolio optimization. We began by acquiring historical stock data, calculating key financial metrics, and understanding inter-asset dynamics through covariance and correlation.

Using Monte Carlo simulations, we explored a wide range of portfolio possibilities, laying the groundwork for more precise optimization. Formal mathematical optimization then allowed us to identify portfolios that precisely maximized the Sharpe Ratio and minimized volatility, providing superior performance characteristics compared to random allocations.

Key takeaways include:

*   **Optimal Portfolio Performance**: The formally optimized Maximum Sharpe Ratio portfolio achieved a significantly higher Sharpe Ratio (1.83) compared to the Monte Carlo approximation, demonstrating the value of rigorous optimization.
*   **Risk-Adjusted Returns**: The backtesting simulation clearly showed the Maximum Sharpe portfolio outperforming both the equally weighted portfolio and the Nifty 50 benchmark, providing superior returns for a given level of risk.
*   **Drawdown Management**: Analysis of drawdowns offered crucial insights into the downside risk characteristics of each portfolio, reinforcing the resilience of the optimized strategies.

This analysis provides a robust framework for making data-driven investment decisions. Future work could explore incorporating more sophisticated risk models, dynamic rebalancing strategies, and machine learning techniques for predictive modeling of asset returns.
```